<a href="https://colab.research.google.com/github/auliatauhid/Data-Science-2026/blob/main/Pertemuan3_AuliaTauhid_250401020136.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Aulia Tauhid Akbar - 250401020136 - Data Science IF405**

In [1]:
# ============================================================
# CELL 1 — IMPORT LIBRARY
# ============================================================

import pandas as pd
import numpy as np
from scipy.stats.mstats import winsorize
import requests

print('✅ Semua library berhasil diimport!')

✅ Semua library berhasil diimport!


In [2]:
# ============================================================
# CELL 2 — MUAT DATASET
# ============================================================

import gdown

file_id = '17YS4lEyhiQNrc2keNclJKbthcyw4t_n9'
output  = 'housing_dirty.csv'

gdown.download(id=file_id, output=output, quiet=False)

df = pd.read_csv(output)

print('✅ Dataset berhasil dimuat!')
print(f'Shape awal: {df.shape}')

Downloading...
From: https://drive.google.com/uc?id=17YS4lEyhiQNrc2keNclJKbthcyw4t_n9
To: /content/housing_dirty.csv
100%|██████████| 4.79k/4.79k [00:00<00:00, 4.86MB/s]

✅ Dataset berhasil dimuat!
Shape awal: (130, 7)


In [3]:
# ============================================================
# CELL 3 — EKSPLORASI AWAL
# ============================================================

print('=== df.info() ===')
df.info()

print('\n=== df.describe() ===')
print(df.describe())

print('\n=== Missing Values ===')
print(df.isnull().sum())

print('\n=== Sampel Data ===')
print(df.head(10))

=== df.info() ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            130 non-null    int64  
 1   luas_m2       112 non-null    float64
 2   harga_juta    113 non-null    float64
 3   kota          130 non-null    object 
 4   kamar         120 non-null    float64
 5   tahun_bangun  130 non-null    int64  
 6   kondisi       130 non-null    object 
dtypes: float64(3), int64(2), object(2)
memory usage: 7.2+ KB

=== df.describe() ===
               id      luas_m2    harga_juta       kamar  tahun_bangun
count  130.000000   112.000000  1.130000e+02  120.000000    130.000000
mean    65.500000   267.627679  8.856325e+05    3.433333   2062.638462
std     37.671829   885.664181  9.407144e+06    1.776283    701.684043
min      1.000000   -50.000000 -5.000000e+02    1.000000   1890.000000
25%     33.250000    87.050000  3.450000e+02    2.0

In [4]:
# ============================================================
# CELL 4 — HAPUS BARIS DUPLIKAT (Masalah M2)
# ============================================================

print(f'Shape sebelum : {df.shape}')
print(f'Jumlah duplikat: {df.duplicated().sum()} baris')

df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'\nShape sesudah  : {df.shape}')
print('✅ Baris duplikat berhasil dihapus!')

Shape sebelum : (130, 7)
Jumlah duplikat: 0 baris

Shape sesudah  : (130, 7)
✅ Baris duplikat berhasil dihapus!


In [5]:
# ============================================================
# CELL 5 — NORMALISASI STRING (Masalah M4)
# ============================================================

print('Sebelum normalisasi:')
print(f"  kota    : {df['kota'].unique()}")
print(f"  kondisi : {df['kondisi'].unique()}")

# 'jakarta' → 'Jakarta', 'BANDUNG' → 'Bandung'
df['kota'] = df['kota'].str.strip().str.title()

# 'Baik' → 'baik', 'SEDANG' → 'sedang'
df['kondisi'] = df['kondisi'].str.strip().str.lower()

print('\nSesudah normalisasi:')
print(f"  kota    : {df['kota'].unique()}")
print(f"  kondisi : {df['kondisi'].unique()}")
print('✅ Normalisasi string selesai!'),

Sebelum normalisasi:
  kota    : ['jogja' 'Medan' 'Depok' 'YGY' 'Jakarta' 'jakarta' 'Yogyakarta' 'Bandung'
 'Surabaya' 'dpk' 'sby' 'Makassar' 'mdn' 'medan' 'Semarang' 'semarang'
 'yogyakarta' 'Jogja' 'JAKARTA' 'Smg' 'DEPOK' 'Bdg' 'makassar' 'surabaya'
 'MAKASSAR' 'depok' 'bandung' 'Bandung ' 'SURABAYA' 'Mksr' ' Jakarta']
  kondisi : ['baik' 'Bagus' 'Sedang' 'baik sekali' 'SEDANG' 'sedang' 'BAIK' 'rusak'
 'cukup' 'Baik' 'Cukup' 'perlu renovasi' 'bagus' 'jelek' 'RUSAK']

Sesudah normalisasi:
  kota    : ['Jogja' 'Medan' 'Depok' 'Ygy' 'Jakarta' 'Yogyakarta' 'Bandung' 'Surabaya'
 'Dpk' 'Sby' 'Makassar' 'Mdn' 'Semarang' 'Smg' 'Bdg' 'Mksr']
  kondisi : ['baik' 'bagus' 'sedang' 'baik sekali' 'rusak' 'cukup' 'perlu renovasi'
 'jelek']
✅ Normalisasi string selesai!


(None,)

In [6]:
# ============================================================
# CELL 6 — IMPUTASI MISSING VALUES (Masalah M1)
# ============================================================

print('Missing values sebelum imputasi:')
print(df.isnull().sum())

# Numerik → isi dengan median
median_luas  = df['luas_m2'].median()
median_harga = df['harga_juta'].median()

df['luas_m2']    = df['luas_m2'].fillna(median_luas)
df['harga_juta'] = df['harga_juta'].fillna(median_harga)

# Kategorik → isi dengan modus
modus_kamar = df['kamar'].mode()[0]
df['kamar'] = df['kamar'].fillna(modus_kamar)

print(f'\nNilai imputasi yang digunakan:')
print(f'  median luas_m2   : {median_luas}')
print(f'  median harga_juta: {median_harga}')
print(f'  modus kamar      : {modus_kamar}')

print('\nMissing values sesudah imputasi:')
print(df.isnull().sum())
print('✅ Imputasi missing values selesai!')

Missing values sebelum imputasi:
id               0
luas_m2         18
harga_juta      17
kota             0
kamar           10
tahun_bangun     0
kondisi          0
dtype: int64

Nilai imputasi yang digunakan:
  median luas_m2   : 193.8
  median harga_juta: 655.0
  modus kamar      : 1.0

Missing values sesudah imputasi:
id              0
luas_m2         0
harga_juta      0
kota            0
kamar           0
tahun_bangun    0
kondisi         0
dtype: int64
✅ Imputasi missing values selesai!


In [7]:
# ============================================================
# CELL 7 — TANGANI OUTLIER IQR FENCE (Masalah M3)
# ============================================================

for col in ['harga_juta', 'luas_m2', 'tahun_bangun']:
    Q1  = df[col].quantile(0.25)
    Q3  = df[col].quantile(0.75)
    IQR = Q3 - Q1
    batas_bawah = Q1 - 1.5 * IQR
    batas_atas  = Q3 + 1.5 * IQR

    outlier = ((df[col] < batas_bawah) | (df[col] > batas_atas)).sum()

    print(f'Kolom [{col}]')
    print(f'  Q1={Q1:.1f} | Q3={Q3:.1f} | IQR={IQR:.1f}')
    print(f'  Batas bawah : {batas_bawah:.1f}')
    print(f'  Batas atas  : {batas_atas:.1f}')
    print(f'  Outlier     : {outlier} baris → di-clip ke batas')
    print()

    df[col] = df[col].clip(lower=batas_bawah, upper=batas_atas)

print('✅ Outlier berhasil ditangani!')

Kolom [harga_juta]
  Q1=380.5 | Q3=916.0 | IQR=535.5
  Batas bawah : -422.8
  Batas atas  : 1719.2
  Outlier     : 3 baris → di-clip ke batas

Kolom [luas_m2]
  Q1=101.6 | Q3=266.1 | IQR=164.5
  Batas bawah : -145.2
  Batas atas  : 513.0
  Outlier     : 1 baris → di-clip ke batas

Kolom [tahun_bangun]
  Q1=1991.2 | Q3=2011.8 | IQR=20.5
  Batas bawah : 1960.5
  Batas atas  : 2042.5
  Outlier     : 3 baris → di-clip ke batas

✅ Outlier berhasil ditangani!


In [8]:
# ============================================================
# CELL 8 — VALIDASI AKHIR & EKSPOR CSV
# ============================================================

# Validasi: harus = 0 semua
assert df.isnull().sum().sum() == 0, '❌ Masih ada missing values!'
assert df.duplicated().sum()   == 0, '❌ Masih ada baris duplikat!'

print('=== VALIDASI AKHIR ===')
print(f'  Missing values : {df.isnull().sum().sum()} ✅')
print(f'  Duplikat       : {df.duplicated().sum()} ✅')
print(f'  Shape akhir    : {df.shape}')

print('\nSampel data bersih:')
print(df.head())

# Ekspor
df.to_csv('housing_clean.csv', index=False)
print('\n✅ Dataset bersih tersimpan → housing_clean.csv')

=== VALIDASI AKHIR ===
  Missing values : 0 ✅
  Duplikat       : 0 ✅
  Shape akhir    : (130, 7)

Sampel data bersih:
   id  luas_m2  harga_juta   kota  kamar  tahun_bangun kondisi
0   1    297.0      1084.0  Jogja    2.0        2000.0    baik
1   2    254.0       761.0  Medan    1.0        1995.0   bagus
2   3    249.7       895.0  Depok    1.0        1983.0    baik
3   4     49.7       178.0    Ygy    5.0        2013.0    baik
4   5    133.4       424.0  Medan    5.0        2004.0  sedang

✅ Dataset bersih tersimpan → housing_clean.csv


In [9]:
# ============================================================
# CELL 9 — AKSES API JSONPLACEHOLDER
# ============================================================

import requests

url_api  = 'https://jsonplaceholder.typicode.com/users'
response = requests.get(url_api)

print(f'Status HTTP: {response.status_code}')

if response.status_code == 200:
    data     = response.json()
    df_users = pd.DataFrame(data)

    # Pilih kolom relevan
    df_users = df_users[['id', 'name', 'username', 'email', 'phone', 'website']]

    print(f'Jumlah data : {len(df_users)} baris')
    print(f'Kolom       : {list(df_users.columns)}')
    print('\nData API:')
    print(df_users.to_string(index=False))

    # Simpan ke CSV
    df_users.to_csv('api_users.csv', index=False)
    print('\n✅ Data API tersimpan → api_users.csv')

else:
    print(f'❌ Gagal. Status: {response.status_code}')

Status HTTP: 200
Jumlah data : 10 baris
Kolom       : ['id', 'name', 'username', 'email', 'phone', 'website']

Data API:
 id                     name         username                     email                 phone       website
  1            Leanne Graham             Bret         Sincere@april.biz 1-770-736-8031 x56442 hildegard.org
  2             Ervin Howell        Antonette         Shanna@melissa.tv   010-692-6593 x09125 anastasia.net
  3         Clementine Bauch         Samantha        Nathan@yesenia.net        1-463-123-4447   ramiro.info
  4         Patricia Lebsack         Karianne Julianne.OConner@kory.org     493-170-9623 x156      kale.biz
  5         Chelsey Dietrich           Kamren  Lucio_Hettinger@annie.ca         (254)954-1289  demarco.info
  6     Mrs. Dennis Schulist Leopoldo_Corkery   Karley_Dach@jasper.info  1-477-935-8478 x6430       ola.org
  7          Kurtis Weissnat     Elwyn.Skiles    Telly.Hoeger@billy.biz          210.067.6132      elvis.io
  8 Nicholas Ru

In [10]:
# ============================================================
# CELL 10 — RINGKASAN AKHIR
# ============================================================

print('=' * 45)
print('         RINGKASAN PIPELINE SELESAI')
print('=' * 45)
print(f'  housing_dirty.csv → housing_clean.csv')
print(f'  Shape akhir       : {df.shape}')
print(f'  Data API users    : {len(df_users)} baris')
print('=' * 45)

         RINGKASAN PIPELINE SELESAI
  housing_dirty.csv → housing_clean.csv
  Shape akhir       : (130, 7)
  Data API users    : 10 baris


**Kesimpulan**

Apa yang Dipelajari
Program ini membangun pipeline data preprocessing lengkap menggunakan Pandas, mencakup: memuat data dari Google Drive dengan gdown, eksplorasi awal (EDA), penanganan 4 masalah kualitas data (missing values, duplikat, outlier, inkonsistensi string), validasi akhir dengan assert, dan konsumsi REST API eksternal menggunakan requests.

Temuan Utama
Kualitas data awal cukup buruk dari 130 baris dataset housing_dirty.csv:
MasalahDetailMissing values18 di luas_m2, 17 di harga_juta, 10 di kamarInkonsistensi kota30+ varian (jogja, YGY, Yogyakarta, JAKARTA, dll.)Outlier ekstremharga_juta sampai 100 juta, tahun_bangun sampai 9999Nilai tidak logisluas_m2 bernilai negatif (-50), harga_juta negatif (-500)
Setelah pipeline selesai, dataset 100% bersih — 0 missing values, 0 duplikat, shape tetap 130×7.

Keterbatasan / Pertanyaan yang Muncul

Normalisasi kota belum tuntas — Ygy, Dpk, Sby, Smg, Bdg, Mdn, Mksr masih tersisa sebagai singkatan yang tidak seragam dengan nama kotanya. Perlu mapping manual (dictionary) agar benar-benar konsisten.
Imputasi dengan median/modus bersifat kasar — mengisi 18 nilai luas_m2 dengan satu angka median (193.8) bisa mendistorsi distribusi. Pendekatan yang lebih baik: imputasi berbasis kelompok kota atau model KNN Imputer.
Kolom kamar: modus = 1.0 terasa aneh — jika dataset properti mayoritas berkamar 4–5, modus 1 mungkin hasil dari data yang sudah rusak sebelum dihitung.
IQR clip vs drop — program memilih clip (menggeser nilai ke batas), bukan membuang baris. Ini menjaga jumlah data tapi "memalsukan" nilai ekstrem menjadi batas IQR.
API JSONPlaceholder hanya untuk latihan — data 10 user tidak terhubung sama sekali dengan dataset housing, jadi belum ada analisis integratif antar keduanya.
Langkah natural berikutnya: feature engineering (harga_per_m2 = harga_juta / luas_m2), encoding kategorikal, lalu modeling regresi untuk prediksi harga.